In [1]:
import pandas as pd
df = pd.read_csv("report.csv")

In [2]:
df.columns

Index(['trace_name', 'uuid', 'memory_requirement', 'cpu_requirement',
       'download_path', 'trace_file', 'slab_size', 'purpose', 'wsr',
       'slab_cnt', 'number_of_requests', 'min_req_size', 'max_req_size', 'qps',
       'number_of_objects', 'number_of_req_GiB', 'number_of_obj_GiB',
       'compulsory_miss_ratio_req', 'compulsory_miss_ratio_byte', 'time_span',
       'frequency_mean', 'file_name', 'wss', 'moveOnSlabRelease',
       'lruRefreshSec', 'poolResizeIntervalSec', 'poolRebalanceIntervalSec',
       'minAllocSize', 'allocFactor', 'rebalanceStrategy', 'cacheSizeMB',
       'maxAllocSize', 'wakeUpRebalancerEveryXReqs', 'mhMovingAverageParam',
       'allocator', 'generator', 'replayGeneratorConfig', 'useTraceTimer',
       'repeatTraceReplay', 'repeatOpCount', 'onlySetIfMiss',
       'enableLookaside', 'timestampFactor', 'numThreads', 'prepopulateCache',
       'zstdTrace', 'compressed', 'traceFileName', 'numOps',
       '_rebalancerNumRuns', '_ramEvictions', '_getMissRatio'

In [18]:
def remap_df(df):
    df = df.copy()

    # 1. Rebalance strategy mapping
    def map_rebalance_strategy(x):
        if x == "marginal-hits-new":
            return "marginal-hits-tuned"
        elif x == "marginal-hits-old":
            return "marginal-hits"
        else:
            return x

    df["rebalance_strategy"] = df["rebalanceStrategy"].apply(map_rebalance_strategy)

    # 2. Allocator mapping
    def map_allocator(x):
        if x == "TINYLFUTail":
            return "TINYLFU"
        elif x == "SIMPLE2Q":
            return "LRU"
        else:
            return x

    df["allocator"] = df["allocator"].apply(map_allocator)

    # 3. Tag mapping
    def map_tag(row):
        if row["rebalanceStrategy"] in ["marginal-hits-new", "marginal-hits-old"] and row["allocator"] == "LRU2Q":
            val = row.get("countColdTailHitsOnly", False)
            # Treat NaN as False
            if pd.notnull(val) and bool(val):
                return "cold-tail"
            else:
                return "warm-cold-tail"
        else:
            return None

    df["tag"] = df.apply(map_tag, axis=1)

    # 4. Rename columns
    rename_dict = {
        "_missRatio": "miss_ratio",
        "_rebalancerNumRebalancedSlabs": "n_rebalanced_slabs",
        "wakeUpRebalancerEveryXReqs": "monitor_interval",
        "_allocFailures": "n_alloc_failures"
    }
    df = df.rename(columns=rename_dict)

    # 5. Select columns to keep
    keep_cols = [
        "trace_name", "number_of_requests", "wsr", "slab_size", "slab_cnt",
        "rebalance_strategy", "allocator", "tag",
        "miss_ratio", "n_rebalanced_slabs", "monitor_interval", "n_alloc_failures", "uuid"
    ]
    # Only keep columns that exist in the DataFrame
    keep_cols = [col for col in keep_cols if col in df.columns]
    return df[keep_cols]

In [20]:
remapped_df = remap_df(df)
remapped_df[remapped_df["trace_name"] == "twitter_cluster53"].sort_values(
    by=["trace_name", "wsr", "miss_ratio", "n_rebalanced_slabs"])

,trace_name,number_of_requests,wsr,slab_size,slab_cnt,rebalance_strategy,allocator,tag,miss_ratio,n_rebalanced_slabs,monitor_interval,n_alloc_failures,uuid
42,twitter_cluster53,246508262,0.05,1,532,marginal-hits-tuned,LRU2Q,cold-tail,0.079619,326,100000,0,twitter_cluster53-5f3193b9e3c89852ecbad6c72291...
24,twitter_cluster53,246508262,0.05,1,532,marginal-hits-tuned,LRU2Q,warm-cold-tail,0.080269,354,100000,0,twitter_cluster53-4411411254a7aaef02204d6a0ebe...
28,twitter_cluster53,246508262,0.05,1,532,marginal-hits,LRU2Q,cold-tail,0.082113,2463,100000,0,twitter_cluster53-4a6f2c978b280edb0a370f0fed43...
2,twitter_cluster53,246508262,0.05,1,532,marginal-hits-tuned,LRU,None,0.082811,331,100000,0,twitter_cluster53-bdec7cb7ce632b31f483cdbb150e...
11,twitter_cluster53,246508262,0.05,1,532,marginal-hits,LRU,None,0.085063,2463,100000,0,twitter_cluster53-3efdf940cf9956ae51e13113d11d...
15,twitter_cluster53,246508262,0.05,1,532,marginal-hits-tuned,TINYLFU,None,0.085184,324,100000,0,twitter_cluster53-694bb076977a026ec0e9a57b6a02...
0,twitter_cluster53,246508262,0.05,1,532,marginal-hits,TINYLFU,None,0.087497,2463,100000,0,twitter_cluster53-b8ebe62cd64ca45a3df3dcf12de4...
4,twitter_cluster53,246508262,0.05,1,532,hits,LRU2Q,None,0.102818,591,100000,0,twitter_cluster53-6ae8eac6ff698dc4b0e3754a6b2f...
35,twitter_cluster53,246508262,0.05,1,532,hits,LRU,None,0.111103,600,100000,0,twitter_cluster53-9e809da26d084cce398fbe57ed19...
17,twitter_cluster53,246508262,0.05,1,532,tail-age,LRU2Q,None,0.143776,651,100000,0,twitter_cluster53-6053348a480c17df4bbfd5578684...


In [ ]:
stats[(stats['trace_name'] == 'meta_202210_kv')].sort_values(by = "_missRatio")
"""
help me write a function, remap the df
rule:
1) rebalanceStrategy == marginal-hits-new -> rebalance_strategy = 'marginal-hits-tuned'
2) rebalanceStrategy == marginal-hits-old -> rebalance_strategy = 'marginal-hits'
other rebalanceStrategy values should be kept as is

allocator 
if TINYLFUTail -> allocator = 'TINYLFU'
if SIMPLE2Q -> allocator = LRU

tag:
if rebalanceStrategy == marginal-hits-new or rebalanceStrategy == marginal-hits-old and allocator = LRU2Q
if countColdTailHitsOnly == True -> tag = 'cold-tail' else tag 'warm-cold-tail'

_missRatio rename to miss_ratio
_rebalancerNumRebalancedSlabs rename to n_rebalanced_slabs
wakeUpRebalancerEveryXReqs rename to monitor_interval
_allocFailures rename to n_alloc_failures


other columns to keep trace_name, number_of_requests, wsr, slab_size, slab_cnt
"""

,trace_name,rebalanceStrategy,allocator,_missRatio,_rebalancerNumRebalancedSlabs,countColdTailHitsOnly,wakeUpRebalancerEveryXReqs
48,meta_202210_kv,marginal-hits-new,LRU2Q,0.034166,1626,True,100000
52,meta_202210_kv,marginal-hits-new,SIMPLE2Q,0.034222,1634,NaN,100000
10,meta_202210_kv,marginal-hits-new,TINYLFUTail,0.034270,1673,NaN,100000
32,meta_202210_kv,marginal-hits-new,LRU2Q,0.034311,2210,NaN,100000
54,meta_202210_kv,hits,LRU2Q,0.034472,2243,NaN,100000
14,meta_202210_kv,hits,SIMPLE2Q,0.034596,2257,NaN,100000
39,meta_202210_kv,hits,TINYLFU,0.034639,2234,NaN,100000
37,meta_202210_kv,marginal-hits-old,SIMPLE2Q,0.036264,14454,NaN,100000
12,meta_202210_kv,marginal-hits-old,TINYLFUTail,0.036340,14454,NaN,100000
6,meta_202210_kv,marginal-hits-old,LRU2Q,0.036966,14454,NaN,100000
